In [ ]:
# ============================================
# RUN THIS FIRST — Colab Setup
# ============================================
!pip install datasets feedparser sentence-transformers -q

# Clone the repo to access project structure
!git clone https://github.com/melissanoarianna1-commits/finpulse-ai.git 2>/dev/null || echo 'Repo already cloned'
import os
os.chdir('/content/finpulse-ai')
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/validated', exist_ok=True)
os.makedirs('models/sentiment', exist_ok=True)
os.makedirs('models/recommender', exist_ok=True)
os.makedirs('docs', exist_ok=True)
print('✅ Setup complete. Working directory:', os.getcwd())

# FinPulse AI — Data Collection & Preparation

This notebook handles **Step 2** of the FinPulse AI pipeline:
1. Download and prepare the **Financial PhraseBank** dataset (for sentiment model)
2. Collect and label **paper abstracts** from arXiv (for recommender model)
3. Run **Great Expectations** validation on both datasets

## Why Data Quality Matters in Financial Risk
In banking, model risk management (Basel III/IV) requires that all model inputs be validated.
Garbage in = garbage out. Our Great Expectations layer enforces this principle programmatically.

## 1. Financial PhraseBank — Sentiment Dataset

The [Financial PhraseBank](https://huggingface.co/datasets/financial_phrasebank) contains ~4,845 English sentences
from financial news, annotated by 5-8 financial domain experts as **positive**, **negative**, or **neutral**.

We use the `sentences_allagree` subset (sentences where all annotators agreed) for highest label quality.
This is the standard benchmark dataset for financial sentiment analysis.

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Download Financial PhraseBank from HuggingFace
dataset = load_dataset('financial_phrasebank', 'sentences_allagree', trust_remote_code=True)
df_sentiment = pd.DataFrame(dataset['train'])

# Map labels: 0=negative, 1=neutral, 2=positive
label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
df_sentiment['label_text'] = df_sentiment['label'].map(label_map)

print(f'Total sentences: {len(df_sentiment)}')
print(f'\nLabel distribution:')
print(df_sentiment['label_text'].value_counts())
print(f'\nSample entries:')
df_sentiment.head()

In [ ]:
# Train/Validation split (80/20, stratified by label)
train_sent, val_sent = train_test_split(
    df_sentiment, test_size=0.2, random_state=42, stratify=df_sentiment['label']
)

print(f'Training set: {len(train_sent)} sentences')
print(f'Validation set: {len(val_sent)} sentences')
print(f'\nTraining label distribution:')
print(train_sent['label_text'].value_counts(normalize=True).round(3))
print(f'\nValidation label distribution:')
print(val_sent['label_text'].value_counts(normalize=True).round(3))

# Save
train_sent.to_csv('data/processed/sentiment_train.csv', index=False)
val_sent.to_csv('data/processed/sentiment_val.csv', index=False)
df_sentiment.to_csv('data/processed/financial_phrasebank.csv', index=False)
print('\n✅ Sentiment data saved to data/processed/')

## 2. Paper Abstracts — Recommender Dataset

For the recommender model, we need paper abstracts labeled as **relevant** or **not relevant**
to Banking & Finance research.

**Strategy:**
- **Relevant papers**: Fetch from arXiv categories `q-fin.RM` (Risk Management), `q-fin.GN` (General Finance), `q-fin.CP` (Computational Finance)
- **Not relevant papers**: Fetch from unrelated arXiv categories like `cs.CV` (Computer Vision), `physics.bio-ph` (Biological Physics)

This gives us a clean binary classification task with naturally labeled data.

In [ ]:
import feedparser
import time

def fetch_arxiv(query, max_results=200):
    url = f'http://export.arxiv.org/api/query?search_query={query}&start=0&max_results={max_results}&sortBy=submittedDate&sortOrder=descending'
    feed = feedparser.parse(url)
    papers = []
    for entry in feed.entries:
        abstract = entry.summary.replace('\n', ' ').strip()
        if len(abstract) > 50:
            papers.append({
                'title': entry.title.replace('\n', ' ').strip(),
                'abstract': abstract,
                'authors': ', '.join([a.name for a in entry.authors]),
                'date': entry.published[:10],
                'source': 'arXiv',
                'url': entry.link,
            })
    return pd.DataFrame(papers)

print('Fetching relevant papers (finance)...')
df_relevant = fetch_arxiv('cat:q-fin.RM+OR+cat:q-fin.GN+OR+cat:q-fin.CP', 200)
df_relevant['relevant'] = 1
print(f'  → {len(df_relevant)} relevant papers')

time.sleep(3)  # Be polite to arXiv API

print('Fetching non-relevant papers (other fields)...')
df_irrelevant = fetch_arxiv('cat:cs.CV+OR+cat:physics.bio-ph+OR+cat:math.CO', 200)
df_irrelevant['relevant'] = 0
print(f'  → {len(df_irrelevant)} non-relevant papers')

# Combine and shuffle
df_papers = pd.concat([df_relevant, df_irrelevant], ignore_index=True)
df_papers = df_papers.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nTotal: {len(df_papers)} papers ({df_papers["relevant"].sum()} relevant, {(df_papers["relevant"]==0).sum()} not relevant)')

In [ ]:
# Train/Val split
train_papers, val_papers = train_test_split(
    df_papers, test_size=0.2, random_state=42, stratify=df_papers['relevant']
)

print(f'Training: {len(train_papers)} | Validation: {len(val_papers)}')

# Save
train_papers.to_csv('data/processed/papers_train.csv', index=False)
val_papers.to_csv('data/processed/papers_val.csv', index=False)
df_papers.to_csv('data/processed/paper_relevance.csv', index=False)
print('✅ Paper data saved to data/processed/')

## 3. Data Validation

We validate both datasets to ensure quality before model training.
This mirrors how banks validate data inputs for risk models under Basel III/IV.

In [ ]:
# ---- Validate Sentiment Dataset ----
print('='*60)
print('VALIDATING: Financial PhraseBank (Sentiment Dataset)')
print('='*60)

checks = {
    'no_null_sentences': train_sent['sentence'].notna().all(),
    'no_null_labels': train_sent['label'].notna().all(),
    'valid_labels': set(train_sent['label'].unique()).issubset({0, 1, 2}),
    'no_empty_sentences': (train_sent['sentence'].str.len() > 0).all(),
    'min_sentence_length': (train_sent['sentence'].str.len() >= 10).all(),
    'no_duplicates': not train_sent.duplicated(subset=['sentence']).any(),
    'all_classes_present': len(train_sent['label'].unique()) == 3,
    'class_balance_ok': train_sent['label'].value_counts(normalize=True).min() > 0.05,
}

for name, passed in checks.items():
    print(f'  {"✅ PASS" if passed else "❌ FAIL"} | {name}')
print(f'\nOverall: {"✅ ALL PASSED" if all(checks.values()) else "❌ SOME FAILED"}')

In [ ]:
# ---- Validate Paper Dataset ----
print('='*60)
print('VALIDATING: Paper Abstracts (Recommender Dataset)')
print('='*60)

checks_p = {
    'no_null_titles': train_papers['title'].notna().all(),
    'no_null_abstracts': train_papers['abstract'].notna().all(),
    'no_null_authors': train_papers['authors'].notna().all(),
    'no_null_dates': train_papers['date'].notna().all(),
    'abstract_min_length': (train_papers['abstract'].str.len() >= 50).all(),
    'no_duplicate_titles': not train_papers.duplicated(subset=['title']).any(),
    'both_classes_present': len(train_papers['relevant'].unique()) == 2,
    'class_balance_ok': train_papers['relevant'].value_counts(normalize=True).min() > 0.3,
}

for name, passed in checks_p.items():
    print(f'  {"✅ PASS" if passed else "❌ FAIL"} | {name}')
print(f'\nOverall: {"✅ ALL PASSED" if all(checks_p.values()) else "❌ SOME FAILED"}')

In [ ]:
# Save validated data
if all(checks.values()):
    train_sent.to_csv('data/validated/sentiment_train.csv', index=False)
    val_sent.to_csv('data/validated/sentiment_val.csv', index=False)
    print('✅ Sentiment data validated and saved')

if all(checks_p.values()):
    train_papers.to_csv('data/validated/papers_train.csv', index=False)
    val_papers.to_csv('data/validated/papers_val.csv', index=False)
    print('✅ Paper data validated and saved')

print('\n--- Step 2 Complete ---')
print('Next: Run 01_sentiment_model.ipynb')